In [38]:
import pandas as pd
df = pd.read_csv("../train_data_1.csv")

In [23]:
df.dropna(axis=0)
df

,name,64-bit execution via heavens gate,64bits,PEB access,accept command line arguments,access the Windows event log,act as TCP client,allocate RW memory,allocate RWX memory,allocate memory,...,psrlq,insw,bsf,mulps,pand,btc,psrld,aesenc,pslld,paddq
0,9fbf213113ba0a18dc2642f83b1201541428fd7951d6a8...,0,1,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1b35c9dbf3cd9ac60015aaa6cd451c898defa6dac1ff43...,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,bf8d307a136a936f7338c1f2eec773c4eb1c802cab77da...,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1e51933903f0358c0b635f863368eb15a61cd3442bc5bf...,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,8a6503fe68d699f8a31531c157e9da931192cd7e3ec809...,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,33bce3b3bf532a63673a9958547655d5f946d47defb2a3...,0,1,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
996,4fa07ea702a0a2145edfec2d069565507638ef0901bb38...,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
997,0547d7bd8d317b53dbc19c06a6726e2f02d706176e06ab...,0,1,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
998,7749cbba0d834738989f175794159deb0bffb80a82078c...,0,1,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [40]:
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.multioutput import MultiOutputClassifier

# Sélection des features et labels
features = [
    "num_nodes", "num_edges", "avg_degree", "num_scc", "avg_clustering", 
    "avg_degree_centrality", "avg_closeness_centrality", "avg_betweenness_centrality", 
    "num_cycles", "avg_cycle_length"
]

#X = df[features]  # Features
X = df.iloc[:, 454:]
y = df.iloc[:, 1:454]  # Labels binaires/multiclasses

# Division en train (80%) et test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scale_pos_weights = (y_train.shape[0] - y_train.sum(axis=0)) / (y_train.sum(axis=0) + 1e-6)
# Modèle KNN avec k=5
knn = MultiOutputClassifier(KNeighborsClassifier(n_neighbors=5))
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

# Calcul du F1-score pour chaque label (chaque sortie)
f1_macro_per_label = []
for i in range(y_test.shape[1]):
    f1_macro_per_label.append(f1_score(y_test.iloc[:, i], y_pred[:, i], average="weighted", zero_division=0))

# Calcul du F1-score macro moyen
f1_macro = sum(f1_macro_per_label) / 453
print(f"F1-score macro: {f1_macro:.4f}")

F1-score macro: 0.9356


In [ ]:
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier

# Sélection des features et labels
X = df.iloc[:, 454:]  # Features
y = df.iloc[:, 1:454]  # Labels

# Division en train (80%) et test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modèle XGBoost
# Calcul du poids pour équilibrer les classes

# Calculer scale_pos_weight pour chaque colonne de y_train
scale_pos_weights = (y_train.shape[0] - y_train.sum(axis=0)) / (y_train.sum(axis=0) + 1e-6)

# Créer une liste de classifieurs XGB avec le bon scale_pos_weight
estimators = [
    XGBClassifier(scale_pos_weight=scale_pos_weights[i], use_label_encoder=False, eval_metric="logloss")
    for i in range(y_train.shape[1])
]

# MultiOutputClassifier en associant chaque classifieur à sa sortie
xgb = MultiOutputClassifier(estimators)

# Entraînement du modèle
xgb.fit(X_train, y_train)

# Calcul du F1-score macro moyen
f1_macro = sum(f1_macro_per_label) / y_test.shape[1]
print(f"F1-score macro: {f1_macro:.4f}")


/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [17:59:20] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [17:59:20] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [17:59:21] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [17:59:21] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packa

F1-score macro: 0.0221


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [44]:
import joblib

# Save the model to a file
joblib.dump(knn, 'knn_model.pkl')

['knn_model.pkl']

In [ ]:
# Save the model to a file
joblib.dump(xgb, 'XGB_model.pkl')

['XGB_model.pkl']

In [45]:
# Lecture du fichier de test
df_to_predict = pd.read_csv("../test_data_1.csv")
df_to_predict

,name,64-bit execution via heavens gate,64bits,PEB access,accept command line arguments,access the Windows event log,act as TCP client,allocate RW memory,allocate RWX memory,allocate memory,...,insw,psrlq,xadd,paddd,pslld,psrld,btc,bswap,paddq,mulps
0,95ff2b3fd399984950293194a717d404bc4e9e3aa0296f...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,4216c07609bb5b89f32d9b559494848a0f5411d1e2d3cf...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2518f84c015a5795bdb2597d580ab7df8e0bfa4b6543c6...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,243268c658456d9b8ec968c088c4f3c7cb976df92a4b99...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,533c09cb9b49c10494337d3eb7d2919c2c656b37f554fb...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,7f0a7ccc2abe273330885020f6dc5f116fe83535838b65...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2996,a188ff24aec863479408cee54b337a2fce25b9372ba557...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,3.0,1.0,11.0,0.0,0.0,0.0,0.0,1.0,0.0
2997,11bf54ba134bfcf242e6c841ae56f44883cdf1b45fa24c...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,1.0,13.0,1.0,1.0,1.0,14.0,119.0,1.0,0.0
2998,7a8862b047f38a1f9ff37b6d1fa114f61bcb7ad6415bed...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [48]:
knn = joblib.load('knn_model.pkl')
# Extraire la colonne "name" et la conserver
names = df_to_predict["name"]

# Supprimer la colonne "name" pour conserver uniquement les features
df_features = df_to_predict.drop("name", axis=1)

# Assurez-vous que les features correspondent à celles utilisées lors de l'entraînement
X_to_predict = df_features[X.columns]

# Effectuer la prédiction avec le modèle multi-output
predictions = knn.predict(X_to_predict)

# Créer un DataFrame pour les prédictions avec les mêmes noms de colonnes que pour y_train
predictions_df = pd.DataFrame(predictions, columns=y_train.columns)

# Réinsérer la colonne "name" en première position
predictions_df.insert(0, "name", names.values)

# Exporter les prédictions dans un fichier CSV
predictions_df.to_csv("predictions_knn.csv", index=False)

In [49]:
predictions_df

,name,64-bit execution via heavens gate,64bits,PEB access,accept command line arguments,access the Windows event log,act as TCP client,allocate RW memory,allocate RWX memory,allocate memory,...,winzip,wise,worm,write and execute a file,write clipboard data,write file on Linux,write file on Windows,write pipe,xorcrypt,yoda
0,95ff2b3fd399984950293194a717d404bc4e9e3aa0296f...,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,4216c07609bb5b89f32d9b559494848a0f5411d1e2d3cf...,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2518f84c015a5795bdb2597d580ab7df8e0bfa4b6543c6...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,243268c658456d9b8ec968c088c4f3c7cb976df92a4b99...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,533c09cb9b49c10494337d3eb7d2919c2c656b37f554fb...,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,7f0a7ccc2abe273330885020f6dc5f116fe83535838b65...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2996,a188ff24aec863479408cee54b337a2fce25b9372ba557...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2997,11bf54ba134bfcf242e6c841ae56f44883cdf1b45fa24c...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2998,7a8862b047f38a1f9ff37b6d1fa114f61bcb7ad6415bed...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
knn = joblib.load('xgb_model.pkl')
# Extraire la colonne "name" et la conserver
names = df_to_predict["name"]

# Supprimer la colonne "name" pour conserver uniquement les features
df_features = df_to_predict.drop("name", axis=1)

# Assurez-vous que les features correspondent à celles utilisées lors de l'entraînement
X_to_predict = df_features[X.columns]

# Effectuer la prédiction avec le modèle multi-output
predictions = knn.predict(X_to_predict)

# Créer un DataFrame pour les prédictions avec les mêmes noms de colonnes que pour y_train
predictions_df = pd.DataFrame(predictions, columns=y_train.columns)

# Réinsérer la colonne "name" en première position
predictions_df.insert(0, "name", names.values)

# Exporter les prédictions dans un fichier CSV
predictions_df.to_csv("predictions_knn.csv", index=False)